In [1]:
import keras
from keras.layers import Conv2D, Conv2DTranspose
from sklearn.model_selection import KFold
import os
import numpy as np
import tensorflow as tf

In [2]:
tf.config.list_physical_devices('GPU')

[]

# Data augmentation

In [3]:
# Adds random horizontal and rotations to our CT scans and labels
# so that more data is added since we have small data.
def get_augmentation():
    return keras.Sequential([
        keras.layers.RandomFlip("horizontal_and_vertical"),
        keras.layers.RandomRotation(0.1),
    ])

# Model Creation

In [ ]:
# Issue: Using a Conv3D may use up a lot of memory which could potentially
# crash the GPU.
def get_model(img_size, num_classes, depth=3):
    # X has shape: (samples, height, width, depth)
    inputs = keras.Input(shape=(*img_size, depth)) # = (1236, 561, 3)

    # Add augmentation
    data_augmentation = get_augmentation()
    x = data_augmentation(inputs)

    # Use padding=same to avoid the creation of a border on feature size.
    # Max pooling is not necessary since we are using a stride of 2 which
    # already downsamples our spatial size.
    x = Conv2D(64, 3, strides=2, activation='relu', padding='same')(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = Conv2D(128, 3, strides=2, activation='relu', padding='same')(x)
    x = Conv2D(128, 3, activation='relu', padding='same')(x)
    x = Conv2D(256, 3, strides=2, activation='relu', padding='same')(x)
    x = Conv2D(256, 3, activation='relu', padding='same')(x)

    # Transpose is uesd to retain the original spatial size and segmentations.
    x = Conv2DTranspose(256, 3, activation='relu', padding='same')(x)
    x = Conv2DTranspose(256, 3, strides=2, activation="relu", padding="same", output_padding=1)(x)
    x = Conv2DTranspose(128, 3, activation="relu", padding="same")(x)
    x = Conv2DTranspose(128, 3, strides=2, activation="relu", padding="same", output_padding=1)(x)
    x = Conv2DTranspose(64, 3, activation="relu", padding="same")(x)
    x = Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same", output_padding=1)(x)

    # Classify each output pixel with our 3 categories
    outputs = Conv2D(num_classes, 1, activation='softmax', padding='same')(x)

    return keras.Model(inputs, outputs)

# Load CT scans with our labels

In [12]:
samples = sorted(os.listdir("../data/Preprocessed/CT Scans"))
print(samples)

['index', 'middle', 'pinky', 'ring']


# Load every CT scan and labels

In [13]:
def load_scan(finger):

    # Load the CT scans and labels for that specific finger (ex. index).
    image_folder = os.path.join("../data/Preprocessed/CT Scans/", finger, "S232028")
    label_folder = os.path.join("../data/Preprocessed/Labels/", finger, "S232028")

    # Ensure they are sorted in order so that the inputs match with their labels.
    image_files = sorted(os.listdir(image_folder))
    label_files = sorted(os.listdir(label_folder))

    images = []
    labels = []

    # Create tuples that contain the CT scans along with their labels.
    for img, lab in zip(image_files, label_files):

        # Both the image_folder and label_folder are loaded using np.load()
        # since they are .npy files, and appends them to their lists.
        images.append(
            np.load(os.path.join(image_folder, img))
        )

        labels.append(
            np.load(os.path.join(label_folder, lab))
        )

    # Converts the arrays from 2D to 3D
    return np.stack(images), np.stack(labels)

# Compute 3 neighboring slices to feed to our model

In [ ]:
def make_multislice(volume, labels, depth=3):

    # Ensure first that the depth is odd so that the center slice is included.
    assert depth % 2 == 1
    pad = depth // 2

    # Add padding along the slice axis.
    # Adds pad slices at the beginning and end of the slice axis.
    vol = np.pad(volume, ((pad, pad), (0, 0), (0, 0)), mode='edge')
    lab = np.pad(labels, ((pad, pad), (0, 0), (0, 0)), mode='edge')

    X = []
    Y = []
    # For each slice index: X gets a stack of depth consecutive slices and Y gets the label of the center slice.
    for i in range(pad, pad + volume.shape[0]):
        X.append(vol[i - pad: i + pad + 1]) # Shape: (depth, height, width)
        Y.append(lab[i]) # Shape: (height, width)
    
    X = np.stack(X) # (samples, depth, height, width)
    Y = np.stack(Y) # (samples, height, width)
    X = np.transpose(X, (0, 2, 3, 1)) # (samples, height, width, depth)
    return X, Y

# Use Kfold cross validation for our small data

In [ ]:
kf = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Max Size: (637, 1236, 561): (slices, height, width)
img_size = (1236, 561) # (height, width)

# This splits into 4 folds, each fold has k-1 CT scans that serves as training
# and one CT scan serves as our testing.
# Each CT scan will serve as our testing.
curr_model = 0
for fold, (train_i, test_i) in enumerate(kf.split(samples)):
    print("Fold:", fold)

    depth = 3 # How many slices the model will take.
    # A new model is created for each fold.
    model = get_model(img_size, 3, depth)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Stores a list of fingers: [x_index, x_middle, x_ring]
    x_train = []
    y_train = []

    # Iterate through each training data in each fold which has a total of 3 fingers.

    for finger in [samples[i] for i in train_i]:
        x, y = load_scan(finger) # returns the inputs associated with their labels.
        x, y = make_multislice(x, y, depth)
        print(x.shape, y.shape)
        x_train.append(x)
        y_train.append(y)
    
    # Concatenates into a shape of (1911, 1236, 561, 3) = (finger_samples, height, width, channels)
    # After concatenation, each slice has shape (1236, 561, 3)
    x_train = np.concatenate(x_train, axis=0)
    y_train = np.concatenate(y_train, axis=0)
    print("x_train: ", x_train.shape)
    print("y_train: ", y_train.shape)

    # Train a new model for each fold.
    # In the end, we average all the accuracies of the models.
    model.fit(
        x_train,
        y_train,
        epochs=10,
        batch_size=1,
    )

    x_test, y_test = load_scan(samples[test_i[0]])
    x_test, y_test = make_multislice(x_test, y_test, depth)

    model.evaluate(x_test, y_test)

    # Save the current model to the 'models' directory. 
    model.save(f'../models/k-fold/model{curr_model}.keras')
    curr_model += 1

Fold: 0
(637, 1236, 561, 3) (637, 1236, 561)
(637, 1236, 561, 3) (637, 1236, 561)
(637, 1236, 561, 3) (637, 1236, 561)
x_train:  (1911, 1236, 561, 3)
y_train:  (1911, 1236, 561)
x_train: (1911, 1236, 561, 3)
y_train: (1911, 1236, 561)


: 